# 🏷️ Auto Tagging Support Tickets Using LLM

## Objective
Automatically tag support tickets into categories using Large Language Models.

This notebook covers:
- Zero-shot classification
- Few-shot prompting
- Fine-tuning a transformer model
- Top-3 tag prediction
- Performance comparison


In [1]:
!pip install transformers datasets scikit-learn pandas numpy accelerate -q

In [2]:
!pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 115.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## 📂 Dataset Loading

### Option 1 (Recommended for Submission)
Place a file named `support_tickets.csv` in the same directory.

Expected format:
| ticket_id | text | label |

### Option 2 (Demo Dataset - Auto Generated)
If CSV not found, a small synthetic dataset will be generated automatically.


In [9]:
import pandas as pd
import os

data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/support_tickets_1500_records.csv')
df = pd.DataFrame(data)

df.head(10)

,ticket_id,text,label
0,1,Support team is not responding. I am frustrated.,Complaint
1,2,Mobile app keeps logging me out. Kindly resolv...,Technical Support
2,3,My account is locked. It has been 3 days already.,Account Access
3,4,Package was damaged during delivery. It has be...,Shipping Problem
4,5,Shipment is delayed. It has been 3 days already.,Shipping Problem
5,6,Please reset my password. It has been 3 days a...,Account Access
6,7,The tracking number is not working. It has bee...,Shipping Problem
7,8,My order has not arrived yet. Kindly resolve A...,Shipping Problem
8,9,This is unacceptable. This is urgent.,Complaint
9,10,Website is not loading properly.,Technical Support


In [10]:
TAGS = [
    'Billing Issue',
    'Technical Support',
    'Account Access',
    'Refund Request',
    'Shipping Problem',
    'Product Inquiry',
    'Complaint',
    'Feature Request'
]

# 🔹 Zero-Shot Classification

In [11]:
from transformers import pipeline

classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

def zero_shot_top3(text):
    result = classifier(text, TAGS)
    labels = result['labels'][:3]
    scores = result['scores'][:3]
    return list(zip(labels, scores))

df['zero_shot_top3'] = df['text'].apply(zero_shot_top3)
df.head()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,ticket_id,text,label,zero_shot_top3
0,1,Support team is not responding. I am frustrated.,Complaint,"[(Complaint, 0.5350236892700195), (Account Acc..."
1,2,Mobile app keeps logging me out. Kindly resolv...,Technical Support,"[(Complaint, 0.5934764742851257), (Account Acc..."
2,3,My account is locked. It has been 3 days already.,Account Access,"[(Account Access, 0.325734406709671), (Complai..."
3,4,Package was damaged during delivery. It has be...,Shipping Problem,"[(Shipping Problem, 0.37047284841537476), (Com..."
4,5,Shipment is delayed. It has been 3 days already.,Shipping Problem,"[(Shipping Problem, 0.8160953521728516), (Comp..."


# 🔹 Fine-Tuning DistilBERT

In [13]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np

dataset = Dataset.from_pandas(df[['text', 'label']])
dataset = dataset.train_test_split(test_size=0.2)

label2id = {label: i for i, label in enumerate(TAGS)}
id2label = {i: label for label, i in label2id.items()}

def encode(example):
    example['label'] = label2id[example['label']]
    return example

dataset = dataset.map(encode)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(TAGS)
)

training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.230039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.19223190640409787, metrics={'train_runtime': 47.5082, 'train_samples_per_second': 50.518, 'train_steps_per_second': 12.629, 'total_flos': 79488943718400.0, 'train_loss': 0.19223190640409787, 'epoch': 2.0})

# 🔹 Top-3 Prediction from Fine-Tuned Model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def fine_tuned_top3(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    top3_idx = np.argsort(probs)[-3:][::-1]

    return [(id2label[i], float(probs[i])) for i in top3_idx]


# 📊 Suggested Public Dataset (Optional)

If you want a real dataset, you may use:

Kaggle: Customer Support Ticket Dataset
https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter

Download CSV and rename it to `support_tickets.csv`.
